# Quick-start: verify the baseline and run one seed
Runs in ~1 minute on CPU. Mirrors scripts/01 and 02 for seed 0, N=199. Full study: `bash scripts/run_all.sh`.

In [1]:
import os, sys, json
os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
from src.data import load_clean, split_queries
from src.methods import BM25, TfidfRetriever, SupervisedLR
from src.metrics import closed_set_metrics
cfg=json.load(open('configs/protocol.json')); C=json.load(open('configs/selected_hparams.json'))['lr_C']
df, desc, rep = load_clean(); print(rep)

{'raw_rows': 20614, 'after_exact_dup_removal': 20563, 'ambiguous_norm_queries_removed': 12, 'after_ambiguity_removal': 20536, 'after_norm_dedup': 20523, 'n_tools': 199, 'all_tools_have_description': True, 'min_q_per_tool': 12, 'max_q_per_tool': 1104, 'median_q_per_tool': 68.0}


In [2]:
tools=sorted(df.Tool.unique()); ix={t:i for i,t in enumerate(tools)}
d=split_queries(df, seed=0); tr,te=d[d.split=='train'],d[d.split=='test']
print(len(tr), len(te), 'text overlap train/test:', len(set(tr.norm)&set(te.norm)))
gold=te.Tool.map(ix).values

12303 4107 text overlap train/test: 0


In [3]:
rows=[]
for m in [BM25(**cfg['bm25']), TfidfRetriever(0), TfidfRetriever(5, seed=0), SupervisedLR(C, cfg['lr']['max_iter'])]:
    m.fit(tools, desc, tr); r=closed_set_metrics(m.score(te.Query), gold); r['method']=m.name; rows.append(r)
pd.DataFrame(rows).set_index('method').round(3)

,top1,top3,mrr,macro_top1,n
method,,,,,
BM25,0.330,0.461,0.418,0.429,4107
TFIDF-desc,0.336,0.467,0.424,0.430,4107
Hybrid(k=5),0.541,0.699,0.638,0.632,4107
TFIDF+LR,0.803,0.908,0.862,0.762,4107


In [4]:
# Look at one real query and the top-3 tools each retrieval method proposes
q=te.iloc[3]; print('QUERY:', q.Query, '| GOLD:', q.Tool)
for m in [BM25(**cfg['bm25']), TfidfRetriever(5, seed=0)]:
    m.fit(tools, desc, tr); s=m.score([q.Query])[0]; print(m.name, [tools[i] for i in np.argsort(-s)[:3]])

QUERY: Can you provide a query tool to search Arxiv? | GOLD: ResearchFinder
BM25 ['HousePurchasingTool', 'HouseRentingTool', 'AI2sql']
Hybrid(k=5) ['ResearchHelper', 'AI2sql', 'fundsdbsearch']
